In [ ]:
%run ../globalvariables

In [ ]:
import numpy as np
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pyspark.sql.functions import lag, avg, stddev
from pyspark.sql.window import Window

In [ ]:
def temporal_split(pdf, date_col="fecha", test_days=TEST_DAYS):
    # Split by date cutoff
    cutoff = pdf[date_col].max() - pd.Timedelta(days=test_days)
    train = pdf[pdf[date_col] <= cutoff]
    test = pdf[pdf[date_col] > cutoff]
    return train, test

In [ ]:
def add_lag_features(df, target_col, partition_cols, lags=(1, 7)):
    # Lags and rolling stats
    w = Window.partitionBy(*partition_cols).orderBy("fecha")
    lag_cols = []
    for n in lags:
        name = f"lag_{n}"
        df = df.withColumn(name, lag(target_col, n).over(w))
        lag_cols.append(name)

    df = df.withColumn("rolling_mean_7d", avg(target_col).over(w.rowsBetween(-6, 0)))
    df = df.withColumn("rolling_mean_30d", avg(target_col).over(w.rowsBetween(-29, 0)))
    df = df.withColumn("rolling_std_7d", stddev(target_col).over(w.rowsBetween(-6, 0)))
    return df.dropna(subset=lag_cols)

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    smape = 100 * np.mean(
        2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)
    )
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "smape": smape,
    }

In [ ]:
def promote_if_better(domain, candidate_run_id, candidate_metric, metric_key="mae"):
    # Promote only if better
    mlflow.set_registry_uri("databricks-uc")
    client = MlflowClient()
    model_name = model_name_for(domain)
    model_uri = f"runs:/{candidate_run_id}/model"
    version = mlflow.register_model(model_uri, model_name).version

    try:
        champion = client.get_model_version_by_alias(model_name, CHAMPION_ALIAS)
        champion_metric = float(client.get_run(champion.run_id).data.metrics[metric_key])
    except Exception:
        champion_metric = None

    if champion_metric is None or candidate_metric < champion_metric:
        client.set_registered_model_alias(model_name, CHAMPION_ALIAS, version)
        return version, True
    return version, False

In [ ]:
def get_champion(model_name):
    # Current champion and MAE
    client = MlflowClient()
    champion = client.get_model_version_by_alias(model_name, CHAMPION_ALIAS)
    champion_mae = float(client.get_run(champion.run_id).data.metrics["mae"])
    return champion, champion_mae

In [ ]:
def predict_batch(features_pdf, model_name, feature_cols):
    # Score with the champion
    model = mlflow.pyfunc.load_model(f"models:/{model_name}@{CHAMPION_ALIAS}")
    return model.predict(features_pdf[feature_cols])